In [2]:
from amplpy import AMPL
import itertools

# --- Generate valid vectors ---
def generate_valid_vectors(N):
    """Generate all 0/1 vectors with the constraint: if vec[i] == 1, then vec[N+i] must be 1"""
    for vec in itertools.product([0, 1], repeat=2*N):
        if all(not (vec[i] == 1 and vec[N+i] == 0) for i in range(N)):
            yield vec

# --- Load AMPL model once ---
ampl = AMPL()
ampl.read("/home/kkishan/Downloads/alan.mod")  # adjust path

# Extract integer/binary variable names
int_var_names = []
for var in ampl.getVariables():
    var_obj = var[1] if isinstance(var, tuple) else var
    int_var_names.append(var_obj.name())

N = len(int_var_names)
print("Number of integer/binary variables:", N)

# --- Solve LP relaxation once ---
ampl.setOption('solver', 'gurobi')
ampl.setOption('gurobi_options', 'NodeLimit=0')  # Only solve LP relaxation (root node)
ampl.solve()

# Print bounds and relaxed values for all variables
for var in ampl.getVariables():
    var_obj = var[1] if isinstance(var, tuple) else var
    print(f"{var_obj.name()} -> LB: {var_obj.lb()}, UB: {var_obj.ub()}, Relaxed solution: {var_obj.value()}")

# --- Iterate over generated vectors, fixing variables and solving ---
results = []

for vec in generate_valid_vectors(N):
    fixed_vars = []

    for i, var_name in enumerate(int_var_names):
        var = ampl.getVariable(var_name)
        lb, ub = var.lb(), var.ub()

        if vec[i] == 1:
            var.fix(lb)
            fixed_vars.append(var)
        elif vec[N+i] == 1:
            var.fix(ub)
            fixed_vars.append(var)

    ampl.setOption('solver', 'gurobi')
    ampl.setOption('gurobi_options', '')  # Remove NodeLimit option for full solve
    ampl.solve()

    obj_val = ampl.getObjective('obj').value()  # Replace 'obj' with your objective name
    results.append((vec, obj_val))

    for var in fixed_vars:
        var.unfix()

# Print results
for vec, val in results:
    print("Vector:", vec, "Objective:", val)


Number of integer/binary variables: 5
Gurobi 12.0.3:   lim:nodes = 0
Gurobi 12.0.3: optimal solution; objective 28
0 simplex iterations
x1 -> LB: 0.0, UB: 1.0, Relaxed solution: 0.0
x2 -> LB: 0.0, UB: 1.0, Relaxed solution: 1.0
x3 -> LB: 0.0, UB: 1.0, Relaxed solution: 1.0
x4 -> LB: 0.0, UB: 1.0, Relaxed solution: 0.0
x5 -> LB: 0.0, UB: 1.0, Relaxed solution: 1.0
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 28
0 simplex iterations
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 28
0 simplex iterations
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 28
0 simplex iterations
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 28
0 simplex iterations
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 28
0 simplex iterations
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 28
0 simplex iterations
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 26
0 simplex iterations
Solution determined by presolve;
objective obj = 26.
Gurobi 12.0.3: